# 01 — Scheduling benchmark: Double DQN versus textbook baselines (notebook-scale run of the reference protocol)

This notebook trains the Double DQN scheduling agent of this repository on the five reporting
seeds (in parallel worker processes),
evaluates it next to the four textbook baselines (FCFS, SJF, Round Robin, Priority) and the
preemptive SRTF reference on the same paired synthetic workloads, plots completion time,
throughput, response time and the training reward, and ends with the scheduling benchmark's
Before / After table and grouped bar chart computed from this notebook's own run.

Everything runs on the discrete-event single-CPU **simulator** in `src/scheduler/` (not a
kernel modification) with synthetic workloads from `configs/scheduler_ddqn.yaml`.

## What this notebook does and does not show

**It shows**

- how the library pieces fit together: `load_scheduler_config`, `make_eval_workloads`,
  `train_agent`, `evaluate_policy`, `evaluate_agent`, and the Before / After helpers in
  `src/common/before_after.py` and `src/common/plotting.py`;
- this notebook's own run: every mean ± standard deviation below is computed over the seeds
  set in the parameters cell (all five reporting seeds by default). If you reduce the seeds,
  the spread is only indicative, and the means can move a long way (see the parameters cell);
- the paired design: every method is evaluated on exactly the same workloads per seed;
- a reproduction check: with no overrides and seeds taken from the reporting seeds 0–4, the
  per-workload values should equal the reference run's rows for those seeds.

**It does not show**

- the documented reference numbers. Those are listed in `experiments/results/RESULTS.md`;
  with the default parameters this notebook re-runs the same protocol and should reproduce
  them, but nothing here replaces or updates them, and this notebook writes nothing under
  `experiments/results/reference/`;
- anything about a real operating system: single simulated CPU, synthetic workloads, every
  workload and agent setting is `[Assumption — not in paper]`;
- a fair learned-versus-heuristic comparison beyond what the oracle allows: the agent and the
  baselines both see the exact length of each task's current CPU burst, which a real operating
  system would have to estimate;
- a benefit of the guardrails: they are evaluated here (agent with and without the three
  override rules). In the reference run the two agents are nearly identical at light and
  medium load, and under heavy load the guarded agent is worse on average, not better.

SRTF (preemptive shortest-remaining-burst-first with the agent's shortest slice as quantum) is
a **reference**, drawn in grey. It is not one of the four Before baselines. It is the like-for-like heuristic for an agent that may preempt, which textbook
SJF may not.

## Parameters

Every knob of this notebook is in the next cell.

In [ ]:
from pathlib import Path

# Configs (paths relative to the repository root).
import os
SCHEDULER_CONFIG = "configs/scheduler_ddqn.yaml"        # simulator, workload, agent, training
BASELINES_CONFIG = "configs/scheduler_baselines.yaml"   # FCFS / SJF / RR / Priority + SRTF reference
BEFORE_AFTER_CONFIG = "configs/before_after.yaml"       # Before / After metric and method definitions

# Dotted overrides applied on top of SCHEDULER_CONFIG, e.g. ["training.episodes=100"].
OVERRIDES = []

# Seeds. Scale choice: all 5 reporting seeds (0-4), the reference training budget
# (training.episodes in the config) and the reference evaluation protocol (10 workloads per
# load profile). Training dominates the run time (under a minute per seed on a laptop CPU)
# and the seeds train in parallel, so the notebook stays within a few minutes.
# Why not fewer seeds: in this benchmark a seed subset is not neutral. The agent's averages are
# driven by rare catastrophic heavy-load workloads that occur on some seeds and not on
# others, so a two-seed run can make the agent look much better (or much worse) than the
# five-seed reference run. For a quick look set SEEDS = [0, 1] and read every result as
# indicative only. Nothing in this notebook is used to choose a setting.
SEEDS = [0, 1, 2, 3, 4]

# Worker processes for training (one seed per process). Results do not depend on it.
WORKERS = max(1, min(len(SEEDS), os.cpu_count() or 1))

# Episodes in the rolling mean of the training-reward plot (display only).
ROLLING_WINDOW = 20

# Output files (git-ignored). Nothing is written under experiments/results/reference/.
OUTPUT_DIR = Path("experiments/results/runs/notebooks/01_scheduler_reproduction")

In [ ]:
import os
from pathlib import Path

# The runner (python -m experiments.run_notebooks) starts the kernel in the repository root.
# If you opened the notebook from the notebooks/ folder instead, move up one level.
if not Path("configs").is_dir() and Path("../configs").is_dir():
    os.chdir("..")
assert Path("configs").is_dir(), "Run this notebook with the repository root as working directory."

from IPython.display import Image, Markdown, display


def provenance(config_paths, overrides, seeds, extra=None):
    """Print where every number below comes from: config file(s), overrides and seeds."""
    paths = [config_paths] if isinstance(config_paths, (str, Path)) else list(config_paths)
    print("config:    " + ", ".join(str(path) for path in paths))
    print("overrides: " + (", ".join(overrides) if overrides else "none"))
    print(f"seeds:     {list(seeds)} ({len(seeds)} seed(s); notebook-scale run, not the reference run)")
    if extra:
        print(extra)


def show_table(header, body, caption=""):
    """Render a small Markdown table (cells are already formatted strings)."""
    lines = ["| " + " | ".join(header) + " |", "|" + "|".join(["---"] * len(header)) + "|"]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in body]
    display(Markdown((caption + "\n\n" if caption else "") + "\n".join(lines)))

In [ ]:
%matplotlib inline
import math
import multiprocessing
import time

import matplotlib.pyplot as plt
import numpy as np

from experiments.build_results import MetricCfg, find_row, measurement
from experiments.run_scheduler_benchmark import DDQN_METHOD, DDQN_UNGUARDED_METHOD, summarise
from src.common import plotting
from src.common.before_after import make_row, render_markdown, write_rows_csv
from src.common.config import load_config
from src.common.io import read_csv, write_csv
from src.common.labels import REPRODUCED
from src.common.metrics import format_mean_std
from src.common.plotting import Bar, BeforeAfterPanel, bar_panel, before_after_figure
from src.scheduler import (
    RAW_EPISODE_FIELDS,
    TRAINING_LOG_FIELDS,
    evaluate_agent,
    evaluate_policy,
    load_baselines_config,
    load_scheduler_config,
    make_policy,
    train_agent,
)
from src.scheduler.training import make_eval_workloads

plotting.apply_style()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_scheduler_config(SCHEDULER_CONFIG, overrides=OVERRIDES)
bcfg = load_baselines_config(BASELINES_CONFIG)
TRACK = load_config(BEFORE_AFTER_CONFIG)["scheduling"]
METRICS = [MetricCfg(**entry) for entry in TRACK["metrics"]]
BY_KEY = {metric.key: metric for metric in METRICS}
LABELS = TRACK["method_labels"]
BEFORE = list(TRACK["before_methods"])
AFTER = TRACK["after_method"]
REFERENCES = list(TRACK["reference_methods"])
ORDER = BEFORE + [AFTER] + REFERENCES
ROLE = {**{m: "before" for m in BEFORE}, AFTER: "after", **{m: "reference" for m in REFERENCES}}
RUN_LABEL = f"notebook-scale run, {len(SEEDS)} seeds"

provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], OVERRIDES, SEEDS)
print(f"training episodes per seed: {cfg.training.episodes} (load profile drawn from {cfg.training.load_profiles})")
print(f"evaluation: {cfg.evaluation.episodes_per_load} workloads per load profile {cfg.evaluation.load_profiles}, epsilon {cfg.evaluation.epsilon}")
shortest = min(s for s in cfg.actions.time_slices_ms if s is not None)
print(f"agent's shortest time slice {shortest} ms; SRTF quantum {bcfg.srtf.quantum_ms} ms")
print("methods:", ", ".join(f"{m} = {LABELS[m]}" for m in ORDER))

## Train the Double DQN agent

One agent per seed, trained online in the simulator on freshly generated training workloads
(child seeds disjoint from the evaluation workloads). The three guardrail override rules are
active during training as configured (`training.guardrails_in_training`); transitions a
guardrail overrode are not stored. If training diverges, the library restores the last
healthy weights and flags it; the cell prints that flag for every seed — check it before
reading any result.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], OVERRIDES, SEEDS, extra=f"workers: {WORKERS}")
started = time.perf_counter()
if WORKERS > 1 and len(SEEDS) > 1:
    with multiprocessing.get_context("spawn").Pool(processes=min(WORKERS, len(SEEDS))) as pool:
        results = pool.starmap(train_agent, [(cfg, seed) for seed in SEEDS], chunksize=1)
else:
    results = [train_agent(cfg, seed) for seed in SEEDS]
training = dict(zip(SEEDS, results))
print(f"trained {len(SEEDS)} agent(s) in {time.perf_counter() - started:.0f} s wall-clock")
for seed, result in training.items():
    reason = f" ({result.divergence_reason})" if result.diverged else ""
    print(f"seed {seed}: {len(result.log)} episodes, {result.agent.env_steps} environment steps, "
          f"{result.wall_seconds:.1f} s, diverged={result.diverged}{reason}")
training_log = [row for result in training.values() for row in result.log]
write_csv(OUTPUT_DIR / "training_log.csv", training_log, TRAINING_LOG_FIELDS)

## Evaluate every method on the same paired workloads

For each seed the evaluation workloads are generated once and every method plays exactly those
workloads: the four baselines and the SRTF reference through `evaluate_policy`, the agent
through `evaluate_agent` with the guardrail overrides on (the After method, `ddqn`) and off
(`ddqn_unguarded`, same network and encoding, a reference). The table pools the light, medium
and heavy load profiles: mean within each seed, then mean ± sample standard deviation across
seeds (`summarise_by_seed`).

How to read it: compare the agent with SJF (the strongest textbook baseline on time metrics)
and with SRTF (the like-for-like reference). Look at the spread as well as the mean: if the
difference between two methods is smaller than their spreads, this run does not separate them.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], OVERRIDES, SEEDS)
rows = []
for seed in SEEDS:
    workloads = make_eval_workloads(cfg, seed)
    for method in [*bcfg.methods, *bcfg.reference_methods]:
        rows.extend(evaluate_policy(cfg, make_policy(method, bcfg), workloads, seed, method))
    agent = training[seed].agent
    rows.extend(evaluate_agent(cfg, agent, workloads, seed, DDQN_METHOD, True))
    rows.extend(evaluate_agent(cfg, agent, workloads, seed, DDQN_UNGUARDED_METHOD, False))
write_csv(OUTPUT_DIR / "raw_episodes.csv", rows, RAW_EPISODE_FIELDS)
summary = summarise(rows)  # per method x load profile, plus load_profile == "all"


def cell(method, key, load="all", digits=None):
    row = find_row(summary, method=method, load_profile=load)
    if row is None:
        return "TBD"
    digits = BY_KEY[key].digits if digits is None and key in BY_KEY else (digits or 2)
    return format_mean_std(row[f"{key}_mean"], row[f"{key}_std"], digits, row["n_seeds"])


table_keys = [m.key for m in METRICS] + ["guardrail_overrides"]
show_table(
    ["Method", "Role"] + [f"{BY_KEY[k].label} ({BY_KEY[k].unit})" if k in BY_KEY else "Guardrail overrides (decisions/episode)" for k in table_keys],
    [[LABELS[m], ROLE[m]] + [cell(m, k) for k in table_keys] for m in ORDER],
    caption=f"**All methods, loads pooled — {RUN_LABEL} {REPRODUCED}.** Mean ± sample standard "
            "deviation across seeds. Reference numbers (five seeds): `experiments/results/RESULTS.md`.",
)

### Reproduction check against the reference run

The simulator, the workloads and the training are deterministic functions of the config and
the seed, and results do not depend on the number of worker processes. So, with **no
overrides** and seeds taken from the reporting seeds 0–4, every per-workload row of this
notebook should equal the reference run's row for the same seed, method, load profile and
workload. The next cell only reads `experiments/results/reference/scheduler/raw_episodes.csv`
and counts matches; it does not print reference values. A mismatch on another machine can come
from a different PyTorch build or CPU (floating-point differences grow during training); it is
not by itself a bug, but it means the notebook's agent is not the reference agent.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], OVERRIDES, SEEDS)
REFERENCE_RAW = Path("experiments/results/reference/scheduler/raw_episodes.csv")
CHECKED = ("avg_turnaround_ms", "avg_response_ms", "throughput_tasks_per_s", "slo_violation_rate")
if OVERRIDES or not set(SEEDS) <= set(cfg.seeds) or not REFERENCE_RAW.is_file():
    print("skipped: the check needs no overrides, seeds from the reporting seeds and the reference CSV")
else:
    reference = {
        (int(r["seed"]), r["method"], r["load_profile"], int(r["episode"])): r
        for r in read_csv(REFERENCE_RAW)
        if int(r["seed"]) in SEEDS
    }
    counts = {}
    for row in rows:
        key = (int(row["seed"]), row["method"], row["load_profile"], int(row["episode"]))
        ref = reference.get(key)
        compared, matched = counts.get(row["method"], (0, 0))
        if ref is None:
            continue
        same = all(math.isclose(float(row[k]), float(ref[k]), rel_tol=1e-9, abs_tol=1e-9) for k in CHECKED)
        counts[row["method"]] = (compared + 1, matched + int(same))
    for method in ORDER:
        compared, matched = counts.get(method, (0, 0))
        print(f"{method:15s} {matched}/{compared} workloads identical to the reference run")

## Completion time, throughput and response time by load

One small panel per metric and load profile; bars start at zero and carry their value; error
bars are the sample standard deviation across seeds. Blue = the four Before baselines, orange =
the agent with guardrails (After), grey = references (SRTF, and the agent with the override
rules off).

How to read it: completion (turnaround) and response time are lower-is-better, throughput is
higher-is-better. Plain throughput differs far less between work-conserving schedulers than
turnaround does (every method eventually finishes the same tasks), so do not read a small
throughput difference as a win. The heavy-load column is where the methods separate most and
where the agent's results vary most from seed to seed.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], OVERRIDES, SEEDS)
PLOT_KEYS = ["avg_turnaround_ms", "throughput_tasks_per_s", "avg_response_ms"]
LOADS = ["all", *cfg.evaluation.load_profiles]
colors = [plotting.ROLE_COLORS[ROLE[m]] for m in ORDER]
fig, axes = plt.subplots(len(PLOT_KEYS), len(LOADS), figsize=(4.6 * len(LOADS), 3.6 * len(PLOT_KEYS)), squeeze=False)
for i, key in enumerate(PLOT_KEYS):
    metric = BY_KEY[key]
    for j, load in enumerate(LOADS):
        found = [find_row(summary, method=m, load_profile=load) for m in ORDER]
        bar_panel(
            axes[i][j], ORDER,
            [r[f"{key}_mean"] if r else math.nan for r in found],
            [r[f"{key}_std"] if r else None for r in found],
            colors,
            ylabel=f"{metric.label} ({metric.unit})",
            title=f"{metric.label}, {'loads pooled' if load == 'all' else load + ' load'}",
            digits=metric.digits,
        )
fig.suptitle(
    f"Scheduling: completion time, throughput and response time by load {REPRODUCED} — {RUN_LABEL}",
    x=0.01, ha="left", fontsize=12, fontweight="bold",
)
fig.text(0.01, 0.002, "Blue = Before baselines, orange = Double DQN with guardrails (After), grey = references. "
         "Synthetic workloads, single simulated CPU; error bars = sample std across seeds.", fontsize=8.5,
         color=plotting.INK_SECONDARY)
fig.tight_layout(rect=(0, 0.02, 1, 0.96))
fig.savefig(OUTPUT_DIR / "methods_by_load.png", dpi=120)
plt.show()

## Training reward

Each panel is one seed: every dot is the return of one training episode (markers show which
load profile the episode drew), and the orange line is a rolling mean. Returns depend strongly
on the load profile of the episode (heavy-load episodes accumulate much larger penalties), so
the dots are spread widely and the rolling mean is noisy.

How to read it: look for the rolling mean rising during the exploration phase and then
levelling off. A curve that keeps falling, or a divergence flag in the training cell, means
the agent you evaluated did not converge. The training return says nothing by itself about
the scheduling metrics: those come only from the evaluation above.

In [ ]:
provenance(SCHEDULER_CONFIG, OVERRIDES, SEEDS, extra=f"rolling window: {ROLLING_WINDOW} episodes")
ncols = min(3, len(SEEDS))
nrows = math.ceil(len(SEEDS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6.0 * ncols, 3.9 * nrows), squeeze=False)
for ax in axes.ravel()[len(SEEDS):]:
    ax.set_visible(False)
for ax, seed in zip(axes.ravel(), SEEDS):
    log = training[seed].log
    episodes = np.array([r["episode"] for r in log])
    returns = np.array([r["episode_return"] for r in log], dtype=float)
    for load, marker in zip(cfg.training.load_profiles, plotting.MARKERS):
        mask = np.array([r["load_profile"] == load for r in log])
        ax.scatter(episodes[mask], returns[mask], s=10, marker=marker, color=plotting.REFERENCE_COLOR,
                   alpha=0.6, label=f"episode return ({load} load)")
    if len(returns) >= ROLLING_WINDOW:
        rolling = np.convolve(returns, np.ones(ROLLING_WINDOW) / ROLLING_WINDOW, mode="valid")
        ax.plot(episodes[ROLLING_WINDOW - 1:], rolling, color=plotting.AFTER_COLOR,
                label=f"rolling mean ({ROLLING_WINDOW} episodes)")
    ax.set_xlabel("training episode")
    ax.set_ylabel("episode return (reward units)")
    ax.set_title(f"seed {seed}", loc="left")
    ax.legend(loc="lower right", fontsize=7.5)
    half = len(returns) // 2
    print(f"seed {seed}: mean return, first half {returns[:half].mean():.2f}, second half {returns[half:].mean():.2f}")
fig.suptitle(f"Double DQN training reward per episode {REPRODUCED} — {RUN_LABEL}",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(OUTPUT_DIR / "training_reward.png", dpi=120)
plt.show()

## Before / After — scheduling benchmark, computed from this notebook's run

The definitions (metrics, units, directions, near-zero thresholds, Before and After methods,
display names) are read from `configs/before_after.yaml`, exactly as
`experiments/build_results.py` reads them for the reference tables; the rows are built with
`make_row` and rendered with `render_markdown` from `src/common/before_after.py`, which apply
the sign convention printed under each table. Before = FCFS, SJF, Round Robin, Priority;
After = the Double DQN agent with guardrails on. SRTF is not a Before method.

How to read it: a positive Improvement % means the agent is better on that metric; a
**negative Improvement % is a regression and is reported as one**. The second table compares
the agent with the best of the four baselines on each metric, which is the most demanding
comparison. This is a notebook-scale table (the number of seeds is printed with it); the
reference Before / After table is in `experiments/results/RESULTS.md`.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG, BEFORE_AFTER_CONFIG], OVERRIDES, SEEDS)
POOLED = TRACK["load_profile"]


def value(method, metric, load=POOLED):
    return measurement(find_row(summary, method=method, load_profile=load), metric)


full = [
    make_row(metric.spec, LABELS[b], value(b, metric), LABELS[AFTER], value(AFTER, metric))
    for metric in METRICS
    for b in BEFORE
]
best = []
for metric in METRICS:
    candidates = [(b, value(b, metric)) for b in BEFORE]
    usable = [(b, m) for b, m in candidates if m.finite]
    pick = (max if metric.higher_is_better else min)(usable, key=lambda item: item[1].mean) if usable else candidates[0]
    best.append(make_row(metric.spec, LABELS[pick[0]], pick[1], LABELS[AFTER], value(AFTER, metric)))

write_rows_csv(full, OUTPUT_DIR / "before_after_scheduling.csv", track="scheduling (notebook-scale)")
display(Markdown(
    f"**Before / After against every baseline — {RUN_LABEL}, loads pooled, source label {REPRODUCED} "
    "(this notebook's run).** Reference numbers: `experiments/results/RESULTS.md`.\n\n" + render_markdown(full)
))
display(Markdown(
    f"**Before / After against the best baseline per metric — {RUN_LABEL}, loads pooled, source label "
    f"{REPRODUCED} (this notebook's run).**\n\n" + render_markdown(best)
))

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG, BEFORE_AFTER_CONFIG], OVERRIDES, SEEDS)
panels = []
for metric in METRICS:
    bars = []
    for method in ORDER:
        measured = value(method, metric)
        bars.append(Bar(LABELS[method], ROLE[method], measured.mean if measured.available else None, measured.std))
    panels.append(BeforeAfterPanel(metric.label, metric.unit, metric.spec.direction, tuple(bars), metric.digits))
chart = before_after_figure(
    panels,
    OUTPUT_DIR / "before_after_scheduling.png",
    f"{TRACK['title']}: Before / After — {RUN_LABEL}",
    REPRODUCED,
    caption=(
        f"Bars: mean across {len(SEEDS)} seeds; error bars: sample standard deviation across seeds "
        "(indicative only with fewer than five seeds). Load profiles pooled. Grey = references that are neither "
        "Before nor After methods. Synthetic workloads; simulator only. Reference run: experiments/results/RESULTS.md."
    ),
)
display(Image(filename=str(chart)))

## Interpretation and limits

- **This is the notebook's own run.** The tables and charts above are means over the seeds set
  in the parameters cell. The documented reference numbers, with their findings, tail analysis
  and ablations, are in `experiments/results/RESULTS.md`; the reproduction check says whether
  this run matched them workload by workload. With fewer than five seeds a standard deviation
  is only indicative, and in this benchmark a seed subset can change which method looks better.
- **Read the Before / After table as it is.** A negative Improvement % is a regression of the
  agent against that baseline, not something to explain away. Check the heavy-load column of
  the by-load figure: in the reference run the agent's heavy-load average is driven by a small
  number of catastrophic workloads, so a seed that happens to contain one can move the pooled
  mean a long way. A good result on a subset of seeds does not rule this out.
- **Compare with SRTF before crediting the learned policy.** The agent may preempt; textbook
  SJF may not. If the agent beats SJF but not SRTF, its advantage over SJF can be explained by
  preemption, which a simple heuristic also has, and the run shows nothing learned beyond it.
- **Guardrails.** Compare `ddqn` and `ddqn_unguarded` and the guardrail-override column. If the
  two are nearly identical, the overrides were an untriggered safety net on these workloads;
  this run does not demonstrate their benefit. If the guarded agent is worse under heavy
  load, as in the reference run, the overrides did not contain the failure and appear to
  have made it worse.
- **Training reward.** A rising training curve shows that the agent optimised its reward; it
  does not show that it schedules better than a baseline.
- **Scope.** Single simulated CPU, synthetic workloads, oracle knowledge of the current burst
  length for every method, every setting `[Assumption — not in paper]`. Nothing here describes
  a real operating system.